# Day 2 · 02 · Performance, Automation and CI/CD Orientation for BI Products

![Lakeflow Job DAG](../../../assets/images/lakeflow_job_dag.png)

How this visual helps: it frames refresh as an orchestrated product workflow, not a manual notebook run. Participants should connect dependencies, triggers and downstream Power BI freshness before the performance sections.

This notebook turns the Power BI dataset into an operational BI product: measurable query performance, cost guardrails, refresh design and deployment awareness.


## Business Scenario

The dataset layer exists, but production BI requires operational discipline. The team must know how to compare query plans, read Query Profile, avoid DirectQuery cost surprises, schedule refreshes and describe deployment with Databricks Asset Bundles.


## Power BI Developer Lens

This module is orientation, not a full automation lab. A Power BI developer should leave knowing how to recognize when a report problem belongs in Power BI, when it belongs in Databricks SQL, and when it should be escalated to the data platform team.

| Symptom | First evidence | Likely owner |
|---|---|---|
| slow page interaction | Power BI Performance Analyzer | report author and semantic-model owner |
| many similar warehouse queries | SQL Warehouse Query History | report author, BI owner |
| broad scans or heavy joins | Query Profile | data model owner / Databricks team |
| stale dataset | refresh history and Lakeflow Job status | refresh owner |
| repeated manual refresh steps | Jobs / DABs orientation | platform or analytics engineering team |

Expected observation: production BI is a shared contract. The Power BI developer does not need to implement every Databricks automation feature, but must know what evidence to bring to the right owner.

## Learning Objectives

By the end of this notebook you can:

- explain Photon and what it does not solve,
- compare detail and aggregate query plans,
- use Query History and Query Profile as evidence,
- apply BI SQL performance rules,
- explain Delta optimization commands safely,
- design Lakeflow Job tasks and DAB deployment orientation,
- explain how new files in a Volume reach Gold, serving objects and Power BI freshness.


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.


In [ ]:
%run ../../../setup/00_setup


### Configuration

Confirm the active catalog and schemas before running the Day 2 examples.


In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))


### Runtime Pre-check

This notebook requires the Workshop 2 dataset/performance layer. Run `solution/w2_powerbi_dataset_solution.ipynb` if this check fails.


In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
    f"{GOLD}.kpi_monthly",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Complete Workshop 2 before running this notebook.")
print("[OK] Workshop 2 serving objects are available.")


## 1. Photon Definition and Limits

**Photon** is Databricks' vectorized query engine for SQL and DataFrame workloads. It can accelerate scans, filters, joins and aggregations.

Photon does not fix poor modeling. If a report sends many DirectQuery visuals to a wide line-grain table, Photon may make each query faster, but it will not remove query fan-out or incorrect grain.


In [ ]:
%sql
SELECT
  'line_grain_detail' AS source,
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_lines
FROM gold.v_fact_sales_incremental
UNION ALL
SELECT
  'monthly_aggregate',
  COUNT(*),
  COUNT(DISTINCT concat_ws('|', year_month, customer_region, category, channel))
FROM gold.fact_sales_dashboard_monthly


## 2. Detail Query vs Aggregate Query

Day1_02 introduced `EXPLAIN`, filter placement, scan reduction and pre-aggregation as BI SQL foundations. This module moves from foundations to production evidence: compare the detail query and serving aggregate, then read Query Profile and Query History to decide what to optimize.


In [ ]:
%sql
EXPLAIN FORMATTED
SELECT
  category,
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
FROM gold.v_fact_sales_incremental
WHERE order_datetime >= TIMESTAMP '2024-01-01 00:00:00'
  AND order_datetime < TIMESTAMP '2025-01-01 00:00:00'
GROUP BY category
ORDER BY revenue DESC


In [ ]:
%sql
EXPLAIN FORMATTED
SELECT
  category,
  ROUND(SUM(revenue), 2) AS revenue
FROM gold.fact_sales_dashboard_monthly
WHERE order_month >= DATE '2024-01-01'
  AND order_month < DATE '2025-01-01'
GROUP BY category
ORDER BY revenue DESC


Expected observation: both queries answer a similar business question, but the aggregate query should have a smaller input shape. This is why W2 created the serving layer.


## 3. Query Profile and Query Insights Workflow

![Query profile reading map](../../../assets/images/query_profile_reading_map.png)

**Query Profile** explains where one query spent work: scan, filter, join, shuffle, aggregate or output.

**Query Insights** is the operational workflow for identifying repeated slow queries, failures and warehouse usage patterns.

Investigation workflow:

1. Open SQL Warehouse Query History.
2. Find the slow visual query by timestamp or user.
3. Open Query Profile.
4. Check scan size, filters, joins and aggregation stages.
5. Compare against the aggregate source created in W2.

### Trainer Demo Query for Screenshots

Run the next SQL cell on a **SQL Warehouse**. It is intentionally shaped like a Power BI visual query: date filter, line-grain scan, `COUNT(DISTINCT)`, grouped KPI calculation and a window rank.

Use it to capture screenshots for this notebook:

- Query History row filtered by tag `day2_02_query_profile_demo`,
- Query Profile graph showing scan, filter, aggregate, window and sort stages,
- Query Insights or history metrics showing duration, rows/bytes read and warehouse used.

Expected observation: this query returns a small result set, but the profile still shows the upstream work needed to produce it. That is the core BI lesson: output rows are not the same thing as compute effort.


In [ ]:
%sql
-- Run on a SQL Warehouse, then open Query History -> Query Profile.
-- Search marker: QI_SCREENSHOT_DEMO_WAREHOUSE_PROFILE
SET QUERY_TAGS['course'] = 'databricks_data_analyst_medi',
    QUERY_TAGS['module'] = 'day2_02_query_profile_demo',
    QUERY_TAGS['demo'] = 'warehouse_profile_screenshot';

/* QI_SCREENSHOT_DEMO_WAREHOUSE_PROFILE */
WITH profile_input AS (
  SELECT
    customer_region,
    category,
    channel,
    order_id,
    line_id,
    quantity,
    line_revenue,
    line_margin,
    order_datetime
  FROM gold.v_fact_sales_incremental
  WHERE order_datetime >= TIMESTAMP '2024-01-01 00:00:00'
    AND order_datetime <  TIMESTAMP '2025-01-01 00:00:00'
    AND is_completed
),
region_category AS (
  SELECT
    customer_region,
    category,
    channel,
    COUNT(*) AS line_rows,
    COUNT(DISTINCT order_id) AS completed_orders,
    ROUND(SUM(quantity), 2) AS quantity,
    ROUND(SUM(line_revenue), 2) AS revenue,
    ROUND(SUM(line_margin), 2) AS gross_margin,
    ROUND(100.0 * SUM(line_margin) / NULLIF(SUM(line_revenue), 0), 2) AS margin_rate_pct
  FROM profile_input
  GROUP BY customer_region, category, channel
),
ranked AS (
  SELECT
    *,
    DENSE_RANK() OVER (
      PARTITION BY customer_region
      ORDER BY revenue DESC
    ) AS revenue_rank_in_region
  FROM region_category
)
SELECT
  current_timestamp() AS profile_run_ts,
  customer_region,
  revenue_rank_in_region,
  category,
  channel,
  line_rows,
  completed_orders,
  quantity,
  revenue,
  gross_margin,
  margin_rate_pct
FROM ranked
WHERE revenue_rank_in_region <= 5
ORDER BY customer_region, revenue_rank_in_region, revenue DESC;


## 4. BI SQL Best Practices Checklist

| Rule | Definition | BI failure prevented |
|---|---|---|
| filter early | reduce rows before grouping | broad scans |
| select columns intentionally | avoid `SELECT *` for serving datasets | wide scans |
| aggregate to visual grain | match table grain to report need | slow visuals |
| protect ratios | use `NULLIF` for denominators | broken KPI cards |
| validate distinct counts | know whether entity count is additive | wrong totals |
| use Query History evidence | tune what actually runs | guess-based optimization |


## 5. Delta Optimization Awareness

`DESCRIBE DETAIL` shows file count and table size. `DESCRIBE HISTORY` shows whether optimization or refresh operations have occurred.


In [ ]:
%sql
DESCRIBE DETAIL gold.fact_sales_dashboard_monthly


In [ ]:
%sql
DESCRIBE HISTORY gold.fact_sales_dashboard_monthly


`OPTIMIZE` compacts small files. `VACUUM ... DRY RUN` lists files eligible for cleanup without deleting them. Use these commands intentionally, not as a blind fix for every slow query.


In [ ]:
%sql
OPTIMIZE gold.fact_sales_dashboard_monthly


In [ ]:
%sql
VACUUM gold.fact_sales_dashboard_monthly RETAIN 168 HOURS DRY RUN


### Optimization Techniques — Overview

As data accumulates, query performance degrades from small files, poor data layout, and full scans. Delta Lake provides four complementary techniques:

| Technique | What it does | When to use |
|-----------|-------------|-------------|
| **OPTIMIZE** | Compacts small Parquet files into larger ones (~1 GB target) | After streaming or frequent small inserts |
| **Partitioning** | Physically separates data into directories by column value | Low-cardinality columns (date, region, status) |
| **Z-ORDER** | Co-locates related data within files for data skipping | High-cardinality filter columns (customer_id, product_id) |
| **Liquid Clustering** | Modern replacement for partitioning + Z-ORDER — incremental, adaptive | New tables or evolving query patterns (recommended) |

**Syntax Reference**

| Command | Syntax |
|---------|--------|
| Compact files | `OPTIMIZE table` |
| Compact + data skipping | `OPTIMIZE table ZORDER BY (col1, col2)` |
| Remove old files | `VACUUM table RETAIN 168 HOURS` |
| Enable Liquid Clustering | `CREATE TABLE t (...) CLUSTER BY (col1, col2)` |
| Change clustering columns | `ALTER TABLE t CLUSTER BY (col3)` — no data rewrite needed |
| Trigger LC compaction | `OPTIMIZE table` (no ZORDER needed) |

### Z-ORDER — Data Skipping

Z-ORDER reorganizes data within files using a multi-dimensional curve. After `OPTIMIZE ... ZORDER BY (col)`, the Delta engine maintains **min/max statistics per file** and skips files that cannot contain the queried value.

| When to use | When NOT to use |
|------------|----------------|
| High-cardinality filter columns (`customer_id`, `product_id`) | Low-cardinality columns — use partitioning instead |
| Up to 4 columns (effectiveness decreases with more) | When Liquid Clustering is available — prefer LC for new tables |

```sql
-- Apply Z-ORDER (combines compaction + data co-location)
OPTIMIZE gold.fact_sales_dashboard_monthly
ZORDER BY (customer_id, category);
```

### Liquid Clustering — Recommended Default

Liquid Clustering is Databricks' modern replacement for partitioning and Z-ORDER. It is **incremental** (each OPTIMIZE applies it progressively) and **adaptive** (clustering columns can change without rewriting data).

| Feature | Partitioning | Z-ORDER | Liquid Clustering |
|---------|-------------|---------|-------------------|
| When to choose | Low-cardinality columns | High-cardinality filter columns | General purpose (recommended) |
| Schema change | Requires full rewrite | Easy to change | Easy to change |
| Maintenance | Manual OPTIMIZE | Manual OPTIMIZE | `OPTIMIZE` (no ZORDER needed) |
| Auto columns | No | No | `CLUSTER BY AUTO` (GA June 2025) |

```sql
-- Create table with Liquid Clustering
CREATE TABLE gold.my_table (...)
CLUSTER BY (customer_id, category);

-- Auto-select clustering columns (GA June 2025)
ALTER TABLE gold.fact_sales_dashboard_monthly CLUSTER BY AUTO;

-- Check clustering columns
DESCRIBE DETAIL gold.fact_sales_dashboard_monthly;  -- see clusteringColumns field
```

### Deletion Vectors

Deletion Vectors make `DELETE`, `UPDATE` and `MERGE` faster by marking rows in a separate bitmap file — instead of rewriting entire Parquet files.

| Aspect | Without Deletion Vectors | With Deletion Vectors |
|--------|------------------------|----------------------|
| DELETE speed | Slow — rewrites affected Parquet files | Fast — marks rows in a vector file |
| Storage during DELETE | Temporary 2x overhead | Minimal overhead |
| Read performance | Normal | Slight overhead (filter step) |
| Cleanup | Automatic | `REORG TABLE ... APPLY (PURGE)` |

```sql
-- Enable on an existing table
ALTER TABLE gold.fact_sales_dashboard_monthly
SET TBLPROPERTIES ('delta.enableDeletionVectors' = true);

-- Purge deletion vectors after bulk DML (optional, improves read speed)
REORG TABLE gold.fact_sales_dashboard_monthly APPLY (PURGE);
```

> **Note:** Deletion Vectors are enabled by default on new Unity Catalog managed tables since DBR 14.1.

### Predictive Optimization

Predictive Optimization uses ML to automatically schedule `OPTIMIZE` and `VACUUM` for Unity Catalog managed tables — eliminating manual maintenance.

| Feature | Manual Maintenance | Predictive Optimization |
|---------|-------------------|------------------------|
| OPTIMIZE | Run manually or schedule | Automatic |
| VACUUM | Run manually or schedule | Automatic |
| Tuning | DBA responsibility | ML-based decisions |
| Default (May 2025+) | Must configure | Enabled by default for UC managed tables |

```sql
-- Enable at schema level
ALTER SCHEMA gold ENABLE PREDICTIVE OPTIMIZATION;

-- Disable for staging/temp tables (avoid unnecessary runs)
ALTER TABLE bronze.staging_temp DISABLE PREDICTIVE OPTIMIZATION;

-- Check optimization history
SELECT * FROM system.storage.predictive_optimization_operations_history
WHERE catalog_name = 'training_dbx_ana'
ORDER BY timestamp DESC;
```

> **For BI pipelines:** With Predictive Optimization active on Gold tables, explicit `OPTIMIZE` steps in the refresh job become optional — Databricks handles compaction automatically.

## 6. Cost Guardrails for BI

Cost is driven by active warehouse time, size, concurrency, data scanned and query fan-out.


### Task Types in Lakeflow Jobs

| Task Type | Description | Use Case |
|-----------|-------------|----------|
| **Notebook** | Run a Databricks notebook | ETL logic, ML training, BI refresh |
| **Pipeline** | Run a Lakeflow Declarative Pipeline | Streaming/batch pipelines |
| **Python Script** | Run a Python file from Git or workspace | Utility scripts, data validation |
| **SQL** | Run a SQL query | DDL, materialized view refresh |
| **If/Else Condition** | Branch based on a condition | Conditional execution (skip/fail paths) |
| **For Each** | Parallel iteration over a list | Fan-out tasks (process multiple tables) |
| **Run Job** | Trigger another Lakeflow Job | Job composability, shared pipelines |
| **JAR / Spark Submit** | Legacy Spark JARs and applications | Migrated Spark workloads |

> **For BI refresh:** Notebook tasks are the standard choice. `For Each` is useful for refreshing multiple Gold tables in parallel.

### Triggers, Retry and Alerting

**Trigger Types**

| Trigger Type | Usage | Example |
|---|---|---|
| **Scheduled** | Fixed schedule (CRON) | `0 0 6 * * ?` — daily at 06:00 Warsaw time |
| **File arrival** | Reaction to new files in UC Volume | New CSV landed in `/Volumes/.../landing/` |
| **Continuous** | Streaming-like continuous processing | Near-real-time pipelines |
| **Manual** | On-demand execution | Testing and one-off runs |

> **Exam Note:** Know CRON syntax (`0 0 6 * * ?` = daily at 06:00) and File Arrival trigger configuration.

**Retry and Alerting**

| Setting | Level | Recommendation |
|---------|-------|----------------|
| Timeout | Task | Set per task — prevents hung tasks |
| Max retries | Task | 2 for transient errors (network, API rate limits) |
| Max concurrent runs | Job | 1 for BI refresh (prevents overlapping runs) |
| Email notifications | Job | On failure — minimum for production |
| Webhooks | Job | Slack/Teams via Notification Destinations |

| Scenario | Retry? | Reason |
|----------|--------|--------|
| Network timeout | Yes | Transient error — retry will succeed |
| API rate limit | Yes | Transient error — backoff helps |
| Data quality failure | No | Retry won't fix bad data |
| Code bug | No | Retry won't fix code |

### Serverless Compute for Jobs

Since 2025, Serverless is the **default compute option** for Lakeflow Jobs. Tasks start in seconds with auto-scaled ephemeral compute — no cluster management needed.

| Aspect | Classic Job Clusters | Serverless Compute |
|--------|----------------------|--------------------|
| Startup time | 3–10 minutes | Seconds |
| Cluster management | Manual (workers, instance types) | Fully automatic |
| Idle cost | Pay while cluster waits | No idle cost |
| DBU pricing | Standard DBUs | Serverless DBUs (slightly higher rate) |
| Best for | Custom Spark tuning, GPU | Most BI refresh and ETL jobs |

> **Default since 2025:** New Jobs default to Serverless. Use classic clusters only when you need specific Spark configurations.

### Widgets and Parameters

Databricks Widgets parameterize notebooks for interactive use in SQL Editor and for passing values from a Job to a notebook task.

| Widget type | Syntax | Use case |
|-------------|--------|----------|
| Text | `dbutils.widgets.text("env", "dev")` | Free-form input |
| Dropdown | `dbutils.widgets.dropdown("region", "EU", ["EU", "US"])` | Select from list |
| Combobox | `dbutils.widgets.combobox("table", "orders", [...])` | Select or type |
| Multiselect | `dbutils.widgets.multiselect("cols", "id", [...])` | Multiple selection |

**Dynamic Job Parameters** — available as widget defaults when a notebook runs as a Job task:

| Parameter | Value |
|-----------|-------|
| `{{job.start_time.iso_date}}` | Run date (YYYY-MM-DD) |
| `{{job.id}}` | Job ID |
| `{{run.id}}` | Current run ID |
| `{{task.name}}` | Current task name |

In [ ]:
# Widget types and usage
dbutils.widgets.text("environment", "dev", "Environment")
dbutils.widgets.dropdown("region", "EU", ["EU", "US", "APAC"], "Region")

# Getting values
environment = dbutils.widgets.get("environment")
region = dbutils.widgets.get("region")

print(f"Environment: {environment}")
print(f"Region: {region}")

# Cleanup
dbutils.widgets.removeAll()

### Monitoring via System Tables

Key table: `system.lakeflow.job_run_timeline` — run history, duration, and result state for all Jobs in the workspace.

| Column | Description |
|--------|-------------|
| `run_name` | Job name |
| `period_start_time` / `period_end_time` | Run window |
| `result_state` | `SUCCESS`, `FAILED`, `CANCELED`, `TIMED_OUT` |
| `job_id` | Unique job identifier |

In [ ]:
%sql
SELECT 
    DATE(period_start_time) AS run_date,
    run_name AS job_name,
    ROUND(
        AVG(
            (UNIX_TIMESTAMP(period_end_time) - UNIX_TIMESTAMP(period_start_time)) / 60
        ), 1
    ) AS avg_duration_min,
    COUNT(*) AS runs,
    COUNTIF(result_state = 'SUCCESS') AS succeeded,
    COUNTIF(result_state != 'SUCCESS') AS failed
FROM system.lakeflow.job_run_timeline
WHERE period_start_time >= current_date() - INTERVAL 14 DAYS
GROUP BY run_date, job_name
ORDER BY run_date DESC

In [ ]:
%sql
WITH guardrails AS (
  SELECT 'Import first' AS guardrail, 'Use Import mode for executive dashboards unless live data is required.' AS rule
  UNION ALL SELECT 'DirectQuery exception', 'Approve DirectQuery only for live operational use cases.'
  UNION ALL SELECT 'Aggregate stable grains', 'Serve common visuals from monthly or daily aggregates.'
  UNION ALL SELECT 'Auto-stop', 'Keep idle SQL Warehouses from running between demos or refreshes.'
  UNION ALL SELECT 'Review Query History', 'Use evidence before resizing a warehouse.'
)
SELECT *
FROM guardrails


## 7. From Volume to Gold to Power BI Freshness

Optional platform demo for Power BI developers.

This is not a full data-engineering lab. It shows the upstream path that explains why a Power BI dataset is fresh, stale or inconsistent:

1. new files land in a Unity Catalog Volume,
2. an incremental ingest step loads only new files,
3. a Job refreshes Gold and Power BI serving objects,
4. Power BI refresh runs after Databricks has published the new state.

Power BI developer lens: when a report does not show new data, ask whether the issue is Power BI refresh, Databricks serving refresh, Gold refresh or file ingestion.

### Volume Landing Path

Unity Catalog Volumes are governed file landing zones. In this course, setup defines:

`DATASET_PATH = /Volumes/{CATALOG}/default/datasets`

For the demo we use:

`DATASET_PATH/incremental_landing/orders/`

The trainer can create two small batch files there. They simulate source-system exports arriving at different times.

In [ ]:
INCREMENTAL_LANDING_PATH = f"{DATASET_PATH}/incremental_landing/orders"

batch_01 = """order_id,order_date,customer_id,product_id,quantity,unit_price,status
900001,2026-01-05,101,1001,2,49.90,Completed
900002,2026-01-05,102,1002,1,129.00,Completed
"""

batch_02 = """order_id,order_date,customer_id,product_id,quantity,unit_price,status
900003,2026-01-06,103,1003,3,19.90,Completed
900004,2026-01-06,104,1004,1,89.00,Returned
"""

dbutils.fs.mkdirs(INCREMENTAL_LANDING_PATH)
dbutils.fs.put(f"{INCREMENTAL_LANDING_PATH}/batch_01.csv", batch_01, overwrite=True)
dbutils.fs.put(f"{INCREMENTAL_LANDING_PATH}/batch_02.csv", batch_02, overwrite=True)

display(spark.createDataFrame([
    ("landing_path", INCREMENTAL_LANDING_PATH),
    ("batch_01", f"{INCREMENTAL_LANDING_PATH}/batch_01.csv"),
    ("batch_02", f"{INCREMENTAL_LANDING_PATH}/batch_02.csv"),
], ["item", "value"]))

dbutils.widgets.text("incremental_landing_path", INCREMENTAL_LANDING_PATH)
print("Widget set: incremental_landing_path")

Expected observation: the files are now in a governed Volume path. This is the part Power BI usually does not see, but it controls whether new source data can reach Gold.

### Incremental Ingest with `COPY INTO`

`COPY INTO` is a SQL-first batch ingestion pattern. It tracks loaded files and skips files that were already processed. This makes it useful for scheduled file drops where new CSV files land in a known folder.

For this course, the code below is a compact orientation demo. A production pipeline would add schema enforcement, data-quality rules and source-system reconciliation.

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS bronze.demo_incremental_orders (
  order_id BIGINT,
  order_date DATE,
  customer_id BIGINT,
  product_id BIGINT,
  quantity INT,
  unit_price DOUBLE,
  status STRING
)
COMMENT 'Orientation demo: incrementally loaded order files from a Unity Catalog Volume.' 

In [ ]:
%sql
COPY INTO bronze.demo_incremental_orders
FROM '${incremental_landing_path}'
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')

In [ ]:
%sql
SELECT
  COUNT(*) AS loaded_rows,
  COUNT(DISTINCT order_id) AS distinct_orders,
  MIN(order_date) AS min_order_date,
  MAX(order_date) AS max_order_date
FROM bronze.demo_incremental_orders

Expected observation: re-running the `COPY INTO` cell should not duplicate files that have already been loaded. This is the key idea behind incremental file ingestion.

Power BI developer lens: if the report is stale, first determine whether new files arrived in the Volume, whether ingestion loaded them, whether Gold refreshed, and whether Power BI refreshed after that.

### Job DAG: Ingest -> Gold -> Serving -> Power BI

A production BI refresh should have an explicit dependency chain.

| Task | Purpose | Owner |
|---|---|---|
| `ingest_new_files` | load new files from Volume to Bronze/Silver | data team |
| `refresh_gold` | rebuild or merge Gold facts and dimensions | data team |
| `refresh_powerbi_serving` | refresh aggregates, incremental views and KPI tables | analytics engineering / BI platform |
| `validate_bi_readiness` | run reconciliation and freshness checks | BI owner + data team |
| `powerbi_dataset_refresh` | refresh Import dataset or validate DirectQuery freshness | Power BI owner |

Important sequencing rule: Power BI refresh should start only after Databricks publishes the expected Gold or serving state.

In [ ]:
%sql
WITH dag AS (
  SELECT 1 AS step_no, 'ingest_new_files' AS task_key, 'Volume CSV -> Bronze/Silver Delta' AS output, 'data team' AS owner
  UNION ALL SELECT 2, 'refresh_gold', 'Gold facts and dimensions', 'data team'
  UNION ALL SELECT 3, 'refresh_powerbi_serving', 'monthly aggregate, incremental view, KPI table', 'analytics engineering / BI platform'
  UNION ALL SELECT 4, 'validate_bi_readiness', 'reconciliation, row counts, freshness checks', 'BI owner + data team'
  UNION ALL SELECT 5, 'powerbi_dataset_refresh', 'Power BI Import refresh or DirectQuery validation', 'Power BI owner'
)
SELECT *
FROM dag
ORDER BY step_no

### Freshness Troubleshooting Checklist

When a Power BI report does not show new data, collect evidence in this order:

| Check | Evidence | Owner to involve |
|---|---|---|
| Did files arrive? | files in `/Volumes/.../incremental_landing/` | source/data team |
| Did ingest load them? | row count and `_source_file` in Bronze/Silver | data team |
| Did Gold refresh? | `DESCRIBE HISTORY gold.fact_sales` and Gold row counts | data team |
| Did W2 serving refresh? | `gold.kpi_monthly`, `gold.fact_sales_dashboard_monthly` freshness | analytics engineering / BI platform |
| Did Power BI refresh after Databricks? | Power BI refresh history | Power BI owner |

Expected observation: freshness is an end-to-end contract. Power BI refresh is only the last step, not the whole pipeline.

## 8. Refresh Strategy and Lakeflow Jobs

A **Lakeflow Job** orchestrates repeatable tasks such as ingesting new files, refreshing Gold tables, validating quality gates and notifying downstream owners.

Recommended BI freshness DAG:

1. Ingest new files from the Volume when source exports arrive.
2. Run Workshop 1 Gold model refresh.
3. Run Workshop 2 serving objects refresh.
4. Run quality and reconciliation checks.
5. Trigger or allow Power BI refresh.
6. Review failures and freshness SLA.

Power BI developer lens: a report refresh should depend on published Databricks data, not race ahead of upstream ingestion or Gold refresh.

### [UI DEMO] BI Refresh Pipeline Job — Ready-to-Use Config

The RetailHub BI freshness pipeline can include an optional ingestion task before the Gold refresh. The required teaching point is the dependency order: Power BI serving objects must wait for Gold, and Power BI refresh must wait for serving validation.

| Task | Notebook or action | Input | Output |
|------|----------|-------|--------|
| **Optional Ingest New Files** | `COPY INTO` / ingestion notebook | Volume landing files | Bronze/Silver Delta rows |
| **W1 Gold Refresh** | `solution/w1_gold_kpi_solution` | `silver.*` | `gold.fact_sales`, `gold.dim_*` |
| **W2 Serving Refresh** | `solution/w2_powerbi_dataset_solution` | `gold.*` (W1 output) | `gold.fact_sales_dashboard_monthly`, `gold.kpi_monthly` |
| **Validate BI Readiness** | checks in this notebook or solution quality gates | Gold + W2 serving | freshness and reconciliation evidence |

**Create the Job — UI Steps:**
1. [ ] Workflows -> Create Job -> Name: `RetailHub_BI_Freshness`
2. [ ] Optional Task 0 — `ingest_new_files` -> Type: Notebook or SQL -> Source: Volume landing path
3. [ ] Add Task 1 — `w1_gold_refresh` -> Type: Notebook -> Path: `/Workspace/Users/<email>/workshops/solution/w1_gold_kpi_solution`
4. [ ] Add Task 2 — `w2_serving_refresh` -> Type: Notebook -> Path: `solution/w2_powerbi_dataset_solution` -> **Depends on:** `w1_gold_refresh`
5. [ ] Add Task 3 — `validate_bi_readiness` -> Type: Notebook or SQL -> **Depends on:** `w2_serving_refresh`
6. [ ] Compute: Serverless when available and approved by workspace policy
7. [ ] Save -> show DAG visualization

**After creating — show participants:**
- [ ] DAG dependency order: ingest -> W1 -> W2 -> validate
- [ ] Task-level timeout and retry policy
- [ ] Job-level max concurrent runs = 1 to prevent overlapping refreshes
- [ ] Run now -> observe sequential execution
- [ ] Failure notification configured for the refresh owner


### DABs YAML — RetailHub BI Refresh Job

Below is the Databricks Asset Bundles definition for the BI refresh pipeline. It includes an optional `ingest_new_files` task to show where Volume ingestion fits in the production chain. In a live project this first task is usually a SQL task with `COPY INTO`, an Auto Loader pipeline or an existing ingestion notebook owned by the data team. Multi-environment targets allow the same bundle to deploy to `dev` and `prod` with different workspace endpoints.

> **Before deploying:** Replace `<your-workspace>`, `<email>` and `<trainer-email>` with actual values.

```yaml
bundle:
  name: retailhub_bi

targets:
  dev:
    mode: development
    workspace:
      host: https://<your-workspace>.azuredatabricks.net
  prod:
    workspace:
      host: https://<prod-workspace>.azuredatabricks.net

resources:
  jobs:
    retailhub_bi_refresh:
      # Optional first task: use a SQL task with COPY INTO, Auto Loader/Lakeflow, or an existing ingestion notebook owned by the data team.
      name: RetailHub_BI_Refresh
      tasks:
        - task_key: ingest_new_files
          notebook_task:
            notebook_path: /Workspace/Users/<email>/<existing_data_team_ingestion_notebook>
            source: WORKSPACE
          job_cluster_key: main_cluster
          timeout_seconds: 900

        - task_key: w1_gold_refresh
          notebook_task:
            notebook_path: /Workspace/Users/<email>/workshops/solution/w1_gold_kpi_solution
            source: WORKSPACE
          depends_on:
            - task_key: ingest_new_files
          job_cluster_key: main_cluster
          timeout_seconds: 1800
          retry_on_timeout: true
          max_retries: 2

        - task_key: w2_serving_refresh
          depends_on:
            - task_key: w1_gold_refresh
          notebook_task:
            notebook_path: /Workspace/Users/<email>/workshops/solution/w2_powerbi_dataset_solution
            source: WORKSPACE
          job_cluster_key: main_cluster
          timeout_seconds: 1800

      job_clusters:
        - job_cluster_key: main_cluster
          new_cluster:
            spark_version: "15.4.x-scala2.12"
            num_workers: 1

      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "Europe/Warsaw"
        pause_status: UNPAUSED

      email_notifications:
        on_failure:
          - <trainer-email>
```


### Deploy Checklist

**Option A — UI (JSON Editor):**
1. [ ] Workflows → Create Job
2. [ ] Click `⚙ Edit as JSON` (top-right)
3. [ ] Paste the JSON definition (replace placeholders)
4. [ ] Save → Run now

**Option B — Databricks CLI:**
```bash
databricks bundle validate
databricks bundle deploy -t dev
databricks bundle run retailhub_bi_refresh -t dev
```

**After deployment — verify:**
- [ ] DAG: optional ingest -> W1 -> W2 -> validate dependency visible in Workflows UI
- [ ] Schedule: daily 06:00 Europe/Warsaw
- [ ] Email alert on failure configured
- [ ] Run succeeds end-to-end: new files ingested if used, Gold tables refreshed, W2 serving layer updated


## 9. DABs Orientation

![DABs deployment flow](../../../assets/images/dabs_deployment_flow.png)

How this visual helps: it shows the production path from local/project files to validated bundle deployment and job execution. The point is orientation: BI serving logic should have a repeatable deployment route, even if this notebook does not deploy it.

**Databricks Asset Bundles** define jobs, notebooks and deployment configuration as versioned project files. This notebook does not deploy; it explains the production handoff.

Typical commands shown for orientation:

```bash
databricks bundle validate
databricks bundle deploy -t dev
databricks bundle run <job-name> -t dev
```

Expected observation: notebooks are training artifacts; production BI refresh should be represented as a job and deployed consistently.


### Git Folders

**Git Folders** (formerly Repos) connect a Databricks workspace folder to a Git repository, enabling version control for notebooks.

**Setup Workflow:**
1. [ ] Workspace → Git Folders → Add a Folder
2. [ ] Enter repository URL (GitHub / GitLab / Azure DevOps)
3. [ ] Choose branch (`main`, `develop`, or feature branch)
4. [ ] Pull to sync latest changes

**Key Operations:**

| Action | Description |
|--------|-------------|
| `git pull` | Sync remote changes to workspace |
| Create branch | Work in isolation, then merge via PR |
| Commit & push | Save notebook changes back to Git |
| Git diff | Compare notebook versions |

> **Exam Tip:** Git Folders enable source control for notebooks directly in the Databricks workspace UI. Changes can be committed without leaving Databricks.

### Target Overrides — Multi-Environment Configuration

DABs support **targets** to configure environment-specific settings. The same bundle deploys to `dev` and `prod` with different workspace endpoints and resource names.

```yaml
bundle:
  name: retailhub_bi

targets:
  dev:
    mode: development      # adds "[dev <username>]" prefix to resource names
    workspace:             # pauses all job triggers automatically
      host: https://adb-dev.azuredatabricks.net
    variables:
      catalog: dev_catalog

  prod:
    mode: production       # live triggers; no prefix; no run-as override
    workspace:
      host: https://adb-prod.azuredatabricks.net
    variables:
      catalog: prod_catalog
```

| Target mode | Behavior |
|-------------|----------|
| `development` | Adds `[dev <username>]` prefix; pauses triggers automatically — safe for testing |
| `production` | No prefix; live triggers; enforces `run_as` identity |

> **Exam Tip:** `mode: development` automatically pauses all job triggers — safe for testing without risk of accidental production runs.

### Databricks CLI — Setup and Authentication

```bash
# Install Databricks CLI (Go-based v0.200+)
brew install databricks/tap/databricks

# Configure authentication
databricks configure --token
# Host: https://adb-<workspace-id>.azuredatabricks.net
# Token: <personal-access-token>

# --- Bundle workflow ---
cd my_bundle/

databricks bundle validate          # Validate YAML syntax
databricks bundle deploy -t dev     # Deploy to dev target
databricks bundle deploy -t prod    # Deploy to production
databricks bundle run retailhub_bi_refresh -t dev  # Run a deployed job
databricks bundle destroy -t dev    # Remove all deployed resources (dev only)
```

### CI Pipeline Integration — GitHub Actions / Azure DevOps

> **[OPTIONAL]** Cover for groups with DevOps or DE background.

```yaml
# .github/workflows/deploy.yml
name: Deploy BI Pipeline

on:
  push:
    branches: [main]

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - name: Install Databricks CLI
        run: pip install databricks-cli
      - name: Deploy bundle
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_TOKEN: ${{ secrets.DATABRICKS_TOKEN }}
        run: databricks bundle deploy --target prod
```

**Pattern:** Push to `main` → CI validates → `bundle deploy --target prod` → resources updated automatically.

| Step | Action |
|------|--------|
| PR opened | CI validates YAML syntax (`bundle validate`) |
| PR merged to `main` | CI deploys to `prod` target |
| Rollback | `git revert` + re-deploy |

## 10. Slow Power BI Report Troubleshooting Path

Use this path when a Databricks-backed report page is slow. The goal is to find evidence before changing model design.

| Step | Tool | Evidence to collect | Typical decision |
|---|---|---|---|
| 1 | Power BI Performance Analyzer | slow visual, DAX time, source query time | reduce visuals or simplify measures |
| 2 | SQL Warehouse Query History | query duration, user, warehouse, repeated query pattern | identify DirectQuery fan-out or refresh load |
| 3 | Query Profile | scan size, joins, shuffle, aggregation stages | filter earlier, reduce columns, pre-aggregate |
| 4 | Gold/serving contract | grain and object purpose | use aggregate table, Import mode or certified view |
| 5 | Operations checklist | warehouse size, auto-stop, schedule overlap | adjust guardrails with platform owner |

Power BI developer lens: do not start by resizing the warehouse. First prove whether the report is asking the right query at the right grain.

## 11. Report Certification Checklist

A report is ready for certification when:

- source objects have documented grain,
- KPI definitions have owners,
- refresh schedule and freshness expectations are documented,
- Query History has no repeated slow visual pattern,
- DirectQuery use is justified,
- warehouse cost guardrails are in place,
- rollback and incident contacts are known.


## Summary

Day2_02 connects performance evidence, dataset shape and operational ownership. Day2_03 can now reuse the same governed serving layer for Databricks-native BI.
